# reactifact quickstart (Colab)

`reactifact.quick` wires the four tasks most projects start with — a structured
call, RAG with citations, an LLM with tools, and a session-persisted chat — in a
few lines, while still building real reactive agents.

It runs **offline with no API key** (honest fallbacks; RAG still retrieves and
cites your documents). Set `OPENROUTER_API_KEY` or `OPENAI_BASE_URL` in the
provider cell to use a real model.

Docs: https://bzdvdn.github.io/reactifact/

In [ ]:
!pip install -q reactifact
# `reactifact.quick` ships on PyPI from 0.10.0 on. Before that, install from git:
# !pip install -q "git+https://github.com/bzdvdn/reactifact"

In [ ]:
from pydantic import BaseModel

from reactifact.quick import agent, rag, tools_agent, chat_agent
from reactifact.tools import tool
from reactifact import Consume, create_agent, produce
from reactifact.quick import Question, Doc

import reactifact
print("reactifact", reactifact.__version__)

## 1. agent — one structured call

`agent(system, schema)` is one LLM call that returns a typed model, materialized
as a reactive `Agent` (so it composes, traces, and re-runs on state).

In [ ]:
class AnswerBody(BaseModel):
    text: str

qa = agent("Answer in one sentence.", AnswerBody)
body = await qa.ask("What are the three states of water?")
print(body.text if body else "(no model configured — set a key in the provider cell below)")

## 2. rag — retrieval with citations (works offline)

`rag(sources)` builds search → materialize → answer, linking the answer
`supported_by` the documents it used. Offline there is no model to phrase the
answer, so we show the retrieved passages and their provenance.

In [ ]:
import pathlib

docs = pathlib.Path("colab_docs")
docs.mkdir(exist_ok=True)
(docs / "refunds.md").write_text("Refunds are issued within 14 days of purchase.")
(docs / "shipping.md").write_text("Standard shipping takes 3 to 5 business days.")

r = rag({"docs": docs})
answer = await r.ask("how do refunds work?")
if answer is not None:
    print(answer.text)
    print("citations:", answer.sources)
else:
    hits = [d.data for d in r.context.list_artifacts(Doc)]
    print("retrieved:", [d.locator for d in hits])
    print(hits[0].text if hits else "(nothing matched)")

## 3. tools — an LLM that can call a function

`tools_agent(system, tools)` wraps `LLMAgent` (`human=True` → `HITLLMAgent`, which
can ask clarifying questions). The tool loop needs a model.

In [ ]:
@tool
async def add(a: float, b: float) -> str:
    """Add two numbers and return the sum."""
    return str(a + b)

t = tools_agent("Use the add tool whenever arithmetic is needed.", [add])
print(await t.ask("What is 1 + 2?") or "(no model — the tool loop needs one)")

## Optional: use a real model

Set **one** provider, then re-run the cells above. `from_env()` tries
`OPENROUTER_API_KEY` first, then any OpenAI-compatible endpoint.

In [ ]:
import os, getpass

# OpenRouter:
# os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ")
# ...or any OpenAI-compatible endpoint (OpenAI, vLLM, Ollama, ...):
# os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
# os.environ["OPENAI_MODEL"] = "gpt-4o-mini"

print(
    "provider configured:",
    bool(os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_BASE_URL")),
)

## 4. chat — sessions

`chat_agent(agents)` returns a configured `ChatAssistant`: sessions, turns,
history. This echo agent needs no model, so it works offline.

In [ ]:
class Reply(BaseModel):
    text: str


@produce(Reply)
async def echo(call):
    question = call.trigger
    if question is None or not isinstance(question.data, Question):
        return None
    call.effects.create(Reply(text=f"you said: {question.data.text}"))
    return None


chat = chat_agent(
    [create_agent("echo", consumes=[Consume(Question)], produces=[echo])]
)
print(await chat.invoke("hello", session_id="demo"))
print(await chat.invoke("again", session_id="demo"))
print("history:", [m["text"] for m in (await chat.history(session_id="demo"))["messages"]])

## Provenance & reproducibility

Every derived artifact links to what it came from, and `audit.context_hash`
fingerprints the whole run — the same state hashes the same (`examples/fintech_audit`).

In [ ]:
from reactifact.audit import context_hash

print("rag run sha256:", context_hash(r.context))

doc = r.context.latest(Doc)
if doc is not None:
    refs = r.context.related(doc.id, "materialized_from")
    print("doc:", doc.data.locator)
    print("  materialized_from:", [x.data.locator for x in refs])

That's the whole on-ramp: `agent` / `rag` / `tools_agent` / `chat_agent`, each
exposing the real `.agent`/`.agents` and the run's `.context` so you graduate to
hand-written `Consume`/`Produce`/`Effects` without rewriting anything.

Next: the [quickstart docs](https://bzdvdn.github.io/reactifact/en/quickstart/) and
the [migration guide](https://bzdvdn.github.io/reactifact/en/migrating/).